## 文档切片 + 混合检索 + 知识库管理 + 性能优化
目标：​ RAG 系统具备工业级能力：能处理长文档、检索精度更高、知识库可维护、响应更快  
  
核心痛点： 
  
Q1.文档切片   

- 之前把整篇文档存入向量库 : "BOM 变更需由工程师在 PLM 系统提交变更申请，经项目经理审批后，通知物料管理员更新物料清单，涉及产线调整的需同步通知生产计划员。"

- 用户问："BOM 变更找谁审批？"
- 检索到整篇文档 → 模型要从一大段里找答案 → 容易漏掉细节  

R1.解决方案：​ 把长文档切成小片段，每段只讲一件事  
  
Q2.检索精度
- 向量检索擅长语义匹配，但不擅长精确匹配
- 用户搜 "PLM 系统" → 向量检索可能找不到，因为语义上 "PLM" 和 "产品生命周期管理" 接近但不是完全匹配 

R2.解决方案：​ 向量检索 + 关键词检索（BM25）混合。

Q3.知识库维护
- 目前知识库是写死在代码里的，不能动态增删改

R3.解决方案：​ 封装知识库管理 API。

## 第 1 步：文档切片策略
新建 document_chunker.py

In [1]:
import re

class DocumentChunker:
    """
    文档切片器：把长文档切成适合检索的小片段
    """
    
    @staticmethod
    def fixed_length_chunk(text, chunk_size=256, overlap=32):
        """
        固定长度切片(按字符数)
        
        参数：
            text:原始文本
            chunk_size:每片字符数
            overlap：相邻片段重叠字符数(保持上下文连贯)
        
        返回：
            切片后的文本列表
        
        """
        chunks = []
        start = 0
        
        while start < len(text):
            end = min(start + chunk_size, len(text))
            chunks.append(text[start:end]) # 左闭右开原则
            start += chunk_size - overlap
        
        return chunks
    
    @staticmethod
    def paragraph_chunk(text, max_chunk_size=512):
        """
        按段落切片(保留段落结构)
        
        参数：
            text:原始文本
            max_chunk_size:最大切片长度
        
        返回：
            [{ "text": "...", "start": 0, "end": 100 }, ...]
        """
        # 按换行符分割段落
        paragraphs = re.split(r"\n\s*\n",text)
        
        chunks = []
        current_chunk = ""
        current_start = 0
        char_offset = 0
        
        for para in paragraphs:
            para = para.strip()
            
            # 空行
            if not para:
                char_offset += 1 # 空行占一个字符
                continue
            
            # 如果当前段落加上去会超长，先把当前 chunk 存起来
            if len(current_chunk) + len(para) > max_chunk_size and current_chunk:
                chunks.append({
                    "text":current_chunk.strip(),
                    "start":current_start,
                    "end":char_offset
                })
                current_chunk = ""
                current_start = char_offset
            
            current_chunk += para + "\n"
            char_offset += len(para) + 1
        
        # 最后一段
        if current_chunk.strip():
            chunks.append({
                "text":current_chunk.strip(),
                "start":current_start,
                "end":char_offset
            })
            
        return chunks
    
    @staticmethod
    def semantic_chunk(text,min_chunk_size=128,max_chunk_size=512):
        """
        语义切片：按句子边界切，尽量保持语义完整
        
        参数：
            text:原始文本
            min_chunk_size: 最小切片长度（低于此值尝试合并）
            max_chunk_size: 最大切片长度（超过此值强制切分）
        
        返回：
            
        """
        # 按句号、问号、感叹号、分号分割
        sentences = re.split(r'(?<=[。！？；\n])', text)
        sentences = [s.strip() for s in sentences if s.strip()] # 删除头尾的空白符，包括空格、\n、\r、\t  ; 判断剩余内容是否非空
        
        chunks = []
        current_chunk = ""
        
        for sentence in sentences:
            # 如果当前句子本身就很长，单独成段
            if len(sentence) > max_chunk_size:
                if current_chunk:
                    chunks.append(current_chunk)
                    current_chunk = ""
                # 对超长句子按逗号进一步切分
                sub_sentences = re.split(r'(?<=[，、])',sentence)
                for sub in sub_sentences:
                    if sub.strip():
                        chunks.append(sub.strip())
                continue
            
            # 尝试合并
            if len(current_chunk) + len(sentence) <= max_chunk_size:
                current_chunk += sentence
            else:
                if current_chunk:
                    chunks.append(current_chunk)
                current_chunk = sentence
        if current_chunk:
            chunks.append(current_chunk)
        
        # 合并太小的片段
        merged_chunks = []
        temp = ""
        for chunk in chunks:
            if len(temp) + len(chunk) <= max_chunk_size and len(temp) < min_chunk_size:
                temp += chunk
            else:
                if temp:
                    merged_chunks.append(temp)
                temp = chunk
        if temp:
            merged_chunks.append(temp)
        
        return merged_chunks
                    
    
    #-------------测试--------------
#     if __name__ == "__main__":
chunker = DocumentChunker()
# 测试长文档
long_doc = """
第一章 总则

第一条 为规范公司员工考勤管理，保障正常工作秩序，特制定本制度。

第二条 本制度适用于公司全体正式员工、试用期员工及实习生。

第三条 人力资源部负责考勤制度的制定与监督执行，各部门负责人负责本部门员工的日常考勤管理。

第二章 工作时间

第四条 标准工作时间为周一至周五 9:00-18:00，午休时间为 12:00-13:30。

第五条 研发部门因项目需要，经部门负责人批准后可实行弹性工作制，但需保证每日工作时长不少于 8 小时。

第三章 迟到与早退

第六条 迟到 30 分钟以内，每次扣款 50 元；迟到超过 30 分钟不足 2 小时的，按半天事假处理；迟到超过 2 小时的，按全天事假处理。

第七条 早退参照迟到标准执行。每月迟到累计 5 次以上，取消当月全勤奖。
"""
print("--------------------------fixed_length_chunk--------------------------")
fixed_chunks = chunker.fixed_length_chunk(long_doc, chunk_size=200, overlap=20)
for i,chunk in enumerate(fixed_chunks):
    print(f"第{i}条：\n{chunk}")
print("--------------------------paragraph_chunk--------------------------")
paragraph_chunks = chunker.paragraph_chunk(long_doc, max_chunk_size=200)
for i,chunk in enumerate(paragraph_chunks):
    print(f"第{i}条：\n{chunk}")
print("--------------------------semantic_chunk--------------------------")
semantic_chunks = chunker.semantic_chunk(long_doc, min_chunk_size=50, max_chunk_size=100)
for i,chunk in enumerate(semantic_chunks):
    print(f"第{i}条：\n{chunk}")

--------------------------fixed_length_chunk--------------------------
第0条：

第一章 总则

第一条 为规范公司员工考勤管理，保障正常工作秩序，特制定本制度。

第二条 本制度适用于公司全体正式员工、试用期员工及实习生。

第三条 人力资源部负责考勤制度的制定与监督执行，各部门负责人负责本部门员工的日常考勤管理。

第二章 工作时间

第四条 标准工作时间为周一至周五 9:00-18:00，午休时间为 12:00-13:30。

第五条 研发部门因项目需要，经部门负责人批
第1条：
五条 研发部门因项目需要，经部门负责人批准后可实行弹性工作制，但需保证每日工作时长不少于 8 小时。

第三章 迟到与早退

第六条 迟到 30 分钟以内，每次扣款 50 元；迟到超过 30 分钟不足 2 小时的，按半天事假处理；迟到超过 2 小时的，按全天事假处理。

第七条 早退参照迟到标准执行。每月迟到累计 5 次以上，取消当月全勤奖。

--------------------------paragraph_chunk--------------------------
第0条：
{'text': '第一章 总则\n第一条 为规范公司员工考勤管理，保障正常工作秩序，特制定本制度。\n第二条 本制度适用于公司全体正式员工、试用期员工及实习生。\n第三条 人力资源部负责考勤制度的制定与监督执行，各部门负责人负责本部门员工的日常考勤管理。\n第二章 工作时间\n第四条 标准工作时间为周一至周五 9:00-18:00，午休时间为 12:00-13:30。', 'start': 0, 'end': 172}
第1条：
{'text': '第五条 研发部门因项目需要，经部门负责人批准后可实行弹性工作制，但需保证每日工作时长不少于 8 小时。\n第三章 迟到与早退\n第六条 迟到 30 分钟以内，每次扣款 50 元；迟到超过 30 分钟不足 2 小时的，按半天事假处理；迟到超过 2 小时的，按全天事假处理。\n第七条 早退参照迟到标准执行。每月迟到累计 5 次以上，取消当月全勤奖。', 'start': 172, 'end': 343}
--------------------------semantic_chunk-

## 第 2 步：重建知识库——使用切片后的文档
新建 rebuild_knowledge_base.py

In [10]:
# 设置 Hugging Face 镜像为国内的 hf-mirror.com
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'  # 报错：网络环境无法访问 Hugging Face 的服务器（被墙了）

import chromadb
from chromadb.utils import embedding_functions

# ---------- 1. 知识库初始化 ----------
client = chromadb.PersistentClient(path="./chroma_db_v2") # 新版知识库，避免冲突
embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection = client.get_or_create_collection(name="oa_knowledge_base_v2",embedding_function=embedding_function)

# ---------- 2. 准备原始文档 ----------
raw_documents = [
    {
        "id": "POL-001",
        "title": "信息安全管理制度",
        "content": """第一章 总则
第一条 为保障公司信息系统安全，防止数据泄露，特制定本制度。
第二条 本制度适用于公司全体员工及外部合作方。

第二章 数据安全管理
第三条 员工不得将公司数据外泄给第三方，不得使用未经授权的软件。
第四条 办公电脑必须安装公司指定的杀毒软件，每周至少进行一次全盘扫描。
第五条 重要数据需加密存储，传输敏感数据时必须使用加密通道。

第三章 账号安全管理
第六条 员工账号密码每月至少更新一次，密码长度不低于8位，需包含大小写字母和数字。
第七条 离职员工账号需在离职当日由IT部门注销。
第八条 违反上述规定者，视情节轻重给予警告、罚款直至开除处分。"""
    },
    {
        "id": "POL-002",
        "title": "考勤管理制度",
        "content": """第一章 总则
第一条 为规范公司员工考勤管理，保障正常工作秩序，特制定本制度。
第二条 本制度适用于公司全体正式员工、试用期员工及实习生。
第三条 人力资源部负责考勤制度的制定与监督执行。

第二章 工作时间
第四条 标准工作时间为周一至周五 9:00-18:00，午休时间为 12:00-13:30。
第五条 研发部门经批准后可实行弹性工作制，但需保证每日工作时长不少于8小时。

第三章 迟到与早退
第六条 迟到30分钟以内扣款50元；超过30分钟按半天事假处理；超过2小时按全天事假处理。
第七条 每月迟到累计5次以上取消当月全勤奖。

第四章 加班管理
第八条 加班需提前在OA提交加班申请，经部门主管审批后生效。
第九条 工作日加班按1.5倍工资计算，休息日加班按2倍工资计算，法定节假日加班按3倍工资计算。"""
    },
    {
        "id": "POL-003",
        "title": "BOM变更管理规范",
        "content": """第一章 总则
第一条 为规范BOM变更流程，确保变更可控可追溯，特制定本规范。
第二条 本规范适用于所有产品的BOM新增、修改、废止操作。

第二章 变更申请
第三条 BOM变更需由工程师在PLM系统提交变更申请，填写变更原因、变更内容、影响范围。
第四条 变更申请需附上技术评审报告和测试验证数据。

第三章 变更审批
第五条 一般变更由项目经理审批，重大变更需经技术总监审批。
第六条 审批时限：项目经理应在3个工作日内完成审批。

第四章 变更执行
第七条 审批通过后，通知物料管理员更新物料清单。
第八条 涉及产线调整的，需同步通知生产计划员调整生产排程。
第九条 变更完成后，工程师需在PLM系统中关闭变更单，归档相关资料。"""
    }
]

# ---------- 3. 切片并存入向量库 ----------
# 清空旧数据
existing_ids= collection.get()["ids"]
if existing_ids:
    collection.delete(ids=existing_ids)

# 文档切片
chunker = DocumentChunker()

all_ids = []
all_chunks = []
all_metadatas = []

for record in raw_documents:
    chunks = chunker.semantic_chunk(record["content"], min_chunk_size=64, max_chunk_size=300)
    
    for i,chunk in enumerate(chunks):
        # 这里存的是切片后各部分对应的id
        chunk_id = f"{record['id']}-chunk-{i:03d}" # d：表示十进制整数 3：表示最小宽度为 3 位 0：表示如果数字不足 3 位，则在左侧用 0 填充
        all_ids.append(chunk_id)
        all_chunks.append(chunk)
        all_metadatas.append({
            "record_id":record["id"],
            "title": record["title"],
            "chunk_index":i,
            "total_chunks":len(chunks) # 切片后被分割为几份
        })
        
# 存入向量库
collection.add(ids=all_ids, documents=all_chunks, metadatas=all_metadatas)

print(f"原始文档数：{len(raw_documents)}")
print(f"切片后文档数：{len(all_chunks)}")
print(f"平均每篇文档切片数：{len(all_chunks)/len(raw_documents):.1f}")
print(all_ids)
print(all_chunks)
print(all_metadatas)
# ---------- 4. 测试检索效果 ----------
test_queries = [
    "BOM变更审批需要多久？",
    "密码有什么要求？",
    "加班费怎么算？",
    "迟到怎么扣钱？",
]

print("\n" + "="*60)
print("测试检索效果（切片后）")
print("="*60)

for query in test_queries:
    print(f"\n查询:{query}")
    results = collection.query(query_texts=[query], n_results=2)
    
    for doc, metadata, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        print(f"  [{metadata['title']} 片段{metadata['chunk_index']}] (相关度：{1-distance:.2f})")
        print(f"  内容：{doc[:120]}...")


原始文档数：3
切片后文档数：5
平均每篇文档切片数：1.7
['POL-001-chunk-000', 'POL-002-chunk-000', 'POL-002-chunk-001', 'POL-003-chunk-000', 'POL-003-chunk-001']
['第一章 总则第一条 为保障公司信息系统安全，防止数据泄露，特制定本制度。第二条 本制度适用于公司全体员工及外部合作方。第二章 数据安全管理第三条 员工不得将公司数据外泄给第三方，不得使用未经授权的软件。第四条 办公电脑必须安装公司指定的杀毒软件，每周至少进行一次全盘扫描。第五条 重要数据需加密存储，传输敏感数据时必须使用加密通道。第三章 账号安全管理第六条 员工账号密码每月至少更新一次，密码长度不低于8位，需包含大小写字母和数字。第七条 离职员工账号需在离职当日由IT部门注销。第八条 违反上述规定者，视情节轻重给予警告、罚款直至开除处分。', '第一章 总则第一条 为规范公司员工考勤管理，保障正常工作秩序，特制定本制度。第二条 本制度适用于公司全体正式员工、试用期员工及实习生。第三条 人力资源部负责考勤制度的制定与监督执行。第二章 工作时间第四条 标准工作时间为周一至周五 9:00-18:00，午休时间为 12:00-13:30。第五条 研发部门经批准后可实行弹性工作制，但需保证每日工作时长不少于8小时。第三章 迟到与早退第六条 迟到30分钟以内扣款50元；超过30分钟按半天事假处理；超过2小时按全天事假处理。第七条 每月迟到累计5次以上取消当月全勤奖。第四章 加班管理第八条 加班需提前在OA提交加班申请，经部门主管审批后生效。', '第九条 工作日加班按1.5倍工资计算，休息日加班按2倍工资计算，法定节假日加班按3倍工资计算。', '第一章 总则第一条 为规范BOM变更流程，确保变更可控可追溯，特制定本规范。第二条 本规范适用于所有产品的BOM新增、修改、废止操作。第二章 变更申请第三条 BOM变更需由工程师在PLM系统提交变更申请，填写变更原因、变更内容、影响范围。第四条 变更申请需附上技术评审报告和测试验证数据。第三章 变更审批第五条 一般变更由项目经理审批，重大变更需经技术总监审批。第六条 审批时限：项目经理应在3个工作日内完成审批。第四章 变更执行第七条 审批通过后，通知物料管理员更

## 第 3 步：混合检索——向量 + 关键词
纯向量检索的短板：精确匹配（如 "PLM 系统"、"OA 提交"）效果不好。  
解决方案：​ 向量检索 + BM25 关键词检索，加权融合

In [8]:
pip install rank-bm25

Note: you may need to restart the kernel to use updated packages.


In [12]:
pip install jieba

Note: you may need to restart the kernel to use updated packages.Collecting jieba
     ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
     ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
      --------------------------------------- 0.3/19.2 MB ? eta -:--:--
     - -------------------------------------- 0.5/19.2 MB 1.7 MB/s eta 0:00:12
     -- ------------------------------------- 1.0/19.2 MB 1.9 MB/s eta 0:00:10
     --- ------------------------------------ 1.6/19.2 MB 2.2 MB/s eta 0:00:08
     ---- ----------------------------------- 2.1/19.2 MB 2.3 MB/s eta 0:00:08
     ------ --------------------------------- 2.9/19.2 MB 2.5 MB/s eta 0:00:07
     ------- -------------------------------- 3.4/19.2 MB 2.5 MB/s eta 0:00:07
     -------- ------------------------------- 4.2/19.2 MB 2.7 MB/s eta 0:00:06
     ---------- ----------------------------- 5.0/19.2 MB 2.8 MB/s eta 0:00:05
     ------------ --------------------------- 5.8/19.2 MB 3.0 MB/s eta

新建 hybrid_retriever.py：

In [15]:
import os
os.environ['hf_endpoint'] = 'https://hf-mirror.com'

import chromadb
from chromadb.utils import embedding_functions
import jieba # 中文分词
from rank_bm25 import BM25Okapi
import numpy as np

class HybridRetriever:
    """
    混合检索器：向量检索 + BM25 关键词检索
    """
    
    def __init__(self):
        # 初始化向量检索
        self.client = chromadb.PersistentClient(path="./chroma_db_v2")
        self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
        self.collection = self.client.get_collection(name="oa_knowledge_base_v2",embedding_function=self.embedding_function)
        
        # 加载所有文档用于BM25
        self._load_all_documents_for_bm25()
        
    def _load_all_documents_for_bm25(self):
        all_data = self.collection.get()
        self.all_ids = all_data["ids"]
        self.all_docs = all_data["documents"]
        self.all_metadatas = all_data["metadatas"]
        
        # 中文分词
        tokenized_docs = []
        for doc in self.all_docs:
            tokenized_docs.append(list(jieba.cut(doc)))
        
        self.bm25 = BM25Okapi(tokenized_docs)
    
    def vector_search(self, query, top_k=5):
        """向量搜索"""
        results = self.collection.query(query_texts=[query], n_results=top_k)
        
        vector_results = []
        for id, doc, metadata, distance in zip(results["ids"][0], results["documents"][0], results["metadatas"][0], results["distances"][0]):
            vector_results.append({
                "id": id,
                "text": doc,
                "metadata": metadata,
                "vector_score": 1 - distance
            })
        return vector_results
    
    def bm25_search(self, query, top_k=5):
        """bm25关键词搜索"""
        tokenized_query = list(jieba.cut(query))
        scores = self.bm25.get_scores(tokenized_query)
        top_indices = np.argsort(scores)[::-1][:top_k] # np.argsort(scores)将 scores 数组从小到大排序所需的‌索引序列;[::-1]对索引数组进行‌反转‌将默认的“升序”转换为‌“降序”;[:top_k]只取反转后数组的前 top_k 个元素。
        
        bm25_results = []
        for index in top_indices:
            if scores[index] > 0:
                bm25_results.append({
                    "id": self.all_ids[index],
                    "text": self.all_docs[index],
                    "metadata": self.all_metadatas[index],
                    "bm25_score": float(scores[index])
                })
        return bm25_results
    
    def hybrid_search(self, query, top_k=3, vector_weight=0.6, bm25_weight=0.4):
        """
        混合检索：加权融合向量检索和BM25关键词检索的结果
        参数：
        返回值：
        """
        # 分别检索
        vector_results = self.vector_search(query,top_k*2)
        bm25_results = self.bm25_search(query, top_k*2)
        
        # 合并得分
        combined_scores = {}
        
        for vr in vector_results:
            doc_id = vr["id"]
            combined_scores[doc_id] = {
                "text":vr["text"],
                "metadata":vr["metadata"],
                "vector_score":vr["vector_score"],
                "bm25_score":0.0
            }
        
        for br in bm25_results:
            doc_id = br["id"]
            if doc_id in combined_scores:
                combined_scores[doc_id]["bm25_score"] = br["bm25_score"]
            else:
                combined_scores[doc_id] = {
                    "text":br["text"],
                    "metadata":br["metadata"],
                    "vector_score":0.0,
                    "bm25_score":br["bm25_score"]
                }
        # 归一化分数并加权
        for doc_id in combined_scores:
            item = combined_scores[doc_id]
            norm_vector = min(item["vector_score"]/1.0, 1.0)
            norm_bm25 = min(item["bm25_score"]/10.0, 1.0) # BM25 分数通常较大
            item["final_score"] = norm_vector * vector_weight + norm_bm25 * bm25_weight 
        
        # 排序取 top_k
        sorted_items = sorted(combined_scores.items(),key=lambda x: x[1]["final_score"],reverse=True)[:top_k] 
        # x代表当前的.item()中迭代的每一个元素：(doc_id, { "text": ..., "vector_score": ..., "bm25_score": ..., "final_score": ... })
        # x[0] 对应元组的第一个元素，即 ‌doc_id‌（文档 ID）。x[1] 对应元组的第二个元素，即‌包含得分和文本信息的字典对象

        results = []
        for doc_id, item in sorted_items:
            results.append({
                "id":doc_id,
                "text":item["text"],
                "metadata":item["metadata"],
                "vector_score":item["vector_score"],
                "bm25_score":item["bm25_score"],
                "final_score":item["final_score"]
            })
        return results

# ------------- 测试 ----------------
# if __name__ == "__main__":
retriever = HybridRetriever()

test_queries = [
        "BOM变更审批时限",
        "密码要求",
        "PLM系统怎么用",
        "加班费计算标准",
        "OA提交离职申请",
    ]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"查询：{query}")
    print(f"{'='*60}")

    # 对比三种检索方式
    print("\n--- 纯向量检索 ---")
    vector_results = retriever.vector_search(query, top_k=2)
    for r in vector_results:
        print(f"  [{r['metadata']['title']}] 得分：{r['vector_score']:.3f}")
        print(f"  {r['text'][:100]}...")

    print("\n--- 纯 BM25 检索 ---")
    bm25_results = retriever.bm25_search(query, top_k=2)
    for r in bm25_results:
        print(f"  [{r['metadata']['title']}] 得分：{r['bm25_score']:.3f}")
        print(f"  {r['text'][:100]}...")

    print("\n--- 混合检索 ---")
    hybrid_results = retriever.hybrid_search(query, top_k=2)
    for r in hybrid_results:
        print(f"  [{r['metadata']['title']}] 综合得分：{r['final_score']:.3f}")
        print(f"  向量分：{r['vector_score']:.3f}, BM25分：{r['bm25_score']:.3f}")
        print(f"  {r['text'][:100]}...")



查询：BOM变更审批时限

--- 纯向量检索 ---
  [BOM变更管理规范] 得分：0.478
  第九条 变更完成后，工程师需在PLM系统中关闭变更单，归档相关资料。...
  [BOM变更管理规范] 得分：0.314
  第一章 总则第一条 为规范BOM变更流程，确保变更可控可追溯，特制定本规范。第二条 本规范适用于所有产品的BOM新增、修改、废止操作。第二章 变更申请第三条 BOM变更需由工程师在PLM系统提交变更申...

--- 纯 BM25 检索 ---
  [BOM变更管理规范] 得分：3.924
  第一章 总则第一条 为规范BOM变更流程，确保变更可控可追溯，特制定本规范。第二条 本规范适用于所有产品的BOM新增、修改、废止操作。第二章 变更申请第三条 BOM变更需由工程师在PLM系统提交变更申...
  [BOM变更管理规范] 得分：0.651
  第九条 变更完成后，工程师需在PLM系统中关闭变更单，归档相关资料。...

--- 混合检索 ---
  [BOM变更管理规范] 综合得分：0.345
  向量分：0.314, BM25分：3.924
  第一章 总则第一条 为规范BOM变更流程，确保变更可控可追溯，特制定本规范。第二条 本规范适用于所有产品的BOM新增、修改、废止操作。第二章 变更申请第三条 BOM变更需由工程师在PLM系统提交变更申...
  [BOM变更管理规范] 综合得分：0.313
  向量分：0.478, BM25分：0.651
  第九条 变更完成后，工程师需在PLM系统中关闭变更单，归档相关资料。...

查询：密码要求

--- 纯向量检索 ---
  [考勤管理制度] 得分：0.407
  第九条 工作日加班按1.5倍工资计算，休息日加班按2倍工资计算，法定节假日加班按3倍工资计算。...
  [BOM变更管理规范] 得分：0.351
  第九条 变更完成后，工程师需在PLM系统中关闭变更单，归档相关资料。...

--- 纯 BM25 检索 ---
  [信息安全管理制度] 得分：0.916
  第一章 总则第一条 为保障公司信息系统安全，防止数据泄露，特制定本制度。第二条 本制度适用于公司全体员工及外部合作方。第二章 数据安全管理第三条 员工不得将公司数据外泄给第三方，不得使用未经授权的软件...



## 第 4 步：知识库管理 API
新建 knowledge_manager.py：

In [25]:
import chromadb
from chromadb.utils import embedding_functions

class KnowledgeManager:
    """
    知识库管理器：增删改查
    """
    def __init__(self, db_path="./chroma_db_v2", collection_name="oa_knowledge_base_v2"):
        self.client = chromadb.PersistentClient(path=db_path)
        self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
        self.collection = self.client.get_collection(name=collection_name, embedding_function=embedding_function)
        
        self.chunker = DocumentChunker()
    
    def add_document(self, doc_id, title, content, chunk_strategy="semantic",chunk_size=300):
        """
        新增一篇文档，自动切片后存入 id title content
        返回：切片数量
        """
        # 切片
        if chunk_strategy == "semantic":
            chunks = self.chunker.semantic_chunk(content,max_chunk_size=chunk_size)
        elif chunk_strategy == "paragraph":
            p_chunks = self.chunker.paragraph_chunk(content, max_chunk_size=chunk_size)
            chunks = [c["text"] for c in p_chunks]
        else:
            chunks = self.chunker.fixed_length_chunk(content, chunk_size=chunk_size)
        
        # 准备数据，ids, metadatas, 已准备documents
        ids = [f"{doc_id}-chunk-{i:03d}" for i in range(len(chunks))]
        metadatas = [{
            "record_id":doc_id,
            "title": title,
            "chunk_index":i,
            "total_chunks":len(chunks) # 切片后被分割为几份
        } for i in range(len(chunks))]
        # 存入
        self.collection.add(ids=ids, documents=chunks, metadatas=metadatas)
        
        print(f"文档 '{title}' 已添加，共 {len(chunks)} 个切片")
        return len(chunks)
    
    def delete_document(self,doc_id):
        """
        删除一篇文档及其所有切片
        """
        # 查询属于该文档的所有切片
        results = self.collection.get(where={"record_id":doc_id})
        
        if not results["ids"]:
            print(f"未找到文档 {doc_id}")
            return 0
        
        # 删除
        self.collection.delete(ids=results["ids"])
        print(f"已删除文档{doc_id}，共{len(results['ids'])}个切片")
        return len(results["ids"])
    
    def update_document(self,doc_id):
        """
        更新一篇文档（先删后加）
        """
        deleted_num = self.delete_document(doc_id)
        added_num = self.add_document(doc_id, title, content)
        print(f"文档 {doc_id} 已更新：删除 {deleted} 个切片，新增 {added} 个切片")
        return added_num
    
    def list_documents(self):
        """
        列出知识库中的所有文档的摘要信息
        """
        all_data = self.collection.get()
        
        # 去重统计
        doc_info = {}
        for meta in all_data["metadatas"]:
            doc_id = meta["record_id"]
            if doc_id not in doc_info:
                doc_info[doc_id] = {
                    "doc_id": doc_id,
                    "title": meta["title"],
                    "chunks": 0
                }
            doc_info[doc_id]["chunks"] += 1
        
        return list(doc_info.values())
    
    def search(self, query, top_k):
        """搜索知识库"""
        results = self.collection.query(query_texts=[query],n_results=top_k)
        
        search_results = []
        for id, doc, metadata, distance in zip(results["ids"][0],results["documents"][0], results["metadatas"][0], results["distances"][0]):
            search_results.append({
                "id":id,
                "text":doc,
                "title":metadata["title"],
                "doc_id":metadata["record_id"],
                "chunk_index":metadata["chunk_index"],
                "relevance":1-distance
            })
        return search_results
    
    def get_statistics(self):
        """获取知识库统计信息"""
        all_data = self.collection.get()
        docs = self.list_documents()
        
        return {
            "total_chunks":len(all_data["ids"]),
            "total_documents":len(docs),
            "documents":docs,
            "average_chunks_per_doc":round(len(all_data["ids"])/len(docs),1) if docs else 0
        }

# ---------- 测试 ----------
# if __name__ == "__main__":
km = KnowledgeManager()

print("=== 知识库统计 ===")
stats = km.get_statistics()
print(f"总文档数：{stats['total_documents']}")
print(f"总切片数：{stats['total_chunks']}")
print(f"平均每篇切片数：{stats['average_chunks_per_doc']}")

print("\n=== 文档列表 ===")
for doc in km.list_documents():
    print(f"  {doc['doc_id']}: {doc['title']} ({doc['chunks']} 个切片)")

print("\n=== 添加新文档 ===")
km.add_document(
    doc_id="POL-004",
    title="会议室管理制度",
    content="""第一章 总则
第一条 为合理利用会议室资源，提高使用效率，特制定本制度。

第二章 预约规则
第二条 会议室通过OA系统预约，单次最长预约4小时。
第三条 预约需提前至少30分钟，取消需提前2小时。

第三章 违规处理
第四条 月度累计爽约3次以上，暂停预约权限一个月。
第五条 会议室使用完毕后需恢复整洁，关闭设备电源。"""
)

print("\n=== 搜索测试 ===")
results = km.search("会议室怎么预约", top_k=2)
for r in results:
    print(f"  [{r['title']}] (相关度：{r['relevance']:.2f})")
    print(f"  {r['text'][:100]}...")

print("\n=== 删除测试文档 ===")
km.delete_document("POL-004")

print("\n=== 最终统计 ===")
stats = km.get_statistics()
print(f"总文档数：{stats['total_documents']}")

=== 知识库统计 ===
总文档数：4
总切片数：6
平均每篇切片数：1.5

=== 文档列表 ===
  POL-001: 信息安全管理制度 (1 个切片)
  POL-002: 考勤管理制度 (2 个切片)
  POL-003: BOM变更管理规范 (2 个切片)
  POL-004: 会议室管理制度 (1 个切片)

=== 添加新文档 ===
文档 '会议室管理制度' 已添加，共 1 个切片

=== 搜索测试 ===
  [会议室管理制度] (相关度：0.46)
  第一章 总则第一条 为合理利用会议室资源，提高使用效率，特制定本制度。第二章 预约规则第二条 会议室通过OA系统预约，单次最长预约4小时。第三条 预约需提前至少30分钟，取消需提前2小时。第三章 违规...
  [考勤管理制度] (相关度：0.45)
  第九条 工作日加班按1.5倍工资计算，休息日加班按2倍工资计算，法定节假日加班按3倍工资计算。...

=== 删除测试文档 ===
已删除文档POL-004，共1个切片

=== 最终统计 ===
总文档数：3


## 第 5 步：性能优化
1 缓存机制  
新建 cached_retriever.py：

In [36]:
import hashlib
import time

class CachedRetriever(HybridRetriever):
    """
    带缓存的混合检索器
    """
    def __init__(self, cache_size=128, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.cache_size = cache_size
        self._cache = {}
        
    def _make_cache_key(self, query, top_k):
        """生成缓存键"""
        return hashlib.md5(f"{query}_{top_k}".encode()).hexdigest()
    
    def hybrid_search(self,query, top_k=3, vector_weight=0.6, bm25_weight=0.4):
        """带缓存的混合检索"""
        cache_key = self._make_cache_key(query, top_k)
        
        if cache_key in self._cache:
            print(f"[缓存命中] {query}")
            return self._cache[cache_key]
        
        # 实际检索
        results = super().hybrid_search(query, top_k, vector_weight, bm25_weight)
        
        # 写入缓存
        if len(self._cache) >= self.cache_size:
            # 简单淘汰：删除最早的一个
            self._cache.pop(next(iter(self._cache)))
        self._cache[cache_key] = results
        
        return results

# 测试缓存效果
# if __name__ == "__main__":
retriever = CachedRetriever()

queries = [
    "BOM变更审批时限",
    "BOM变更审批时限",  # 重复查询，应该命中缓存
    "密码要求",
    "BOM变更审批时限",  # 再次命中
]

for q in queries:
    start = time.time()
    results = retriever.hybrid_search(q, top_k=2)
    elapsed = time.time() - start
    print(f"查询耗时：{elapsed:.3f}s")
    print(f"结果：{results[0]['text'][:60] if results else '无'}...\n")

查询耗时：0.019s
结果：第一章 总则第一条 为规范BOM变更流程，确保变更可控可追溯，特制定本规范。第二条 本规范适用于所有产品的BOM新增、修...

[缓存命中] BOM变更审批时限
查询耗时：0.000s
结果：第一章 总则第一条 为规范BOM变更流程，确保变更可控可追溯，特制定本规范。第二条 本规范适用于所有产品的BOM新增、修...

查询耗时：0.018s
结果：第九条 工作日加班按1.5倍工资计算，休息日加班按2倍工资计算，法定节假日加班按3倍工资计算。...

[缓存命中] BOM变更审批时限
查询耗时：0.000s
结果：第一章 总则第一条 为规范BOM变更流程，确保变更可控可追溯，特制定本规范。第二条 本规范适用于所有产品的BOM新增、修...



5.2 批量检索优化

In [37]:
def batch_search(self, queries, top_k):
    """
    批量检索多条查询，减少 API 调用次数
    
    参数：
        queries: 查询列表
        top_k: 每条查询返回的文档数
    
    返回：
        每个查询对应的检索结果列表
    """
    # Chroma 原生支持批量查询
    results = self.collection.query(query_texts=queries,n_results=top_k)
    
    batch_results = []
    for i in len(queries):
        query_results = []
        for doc, metadata, distance in zip(results["documents"][i], results["metadatas"][i], results["distances"][i]):
            query_results.append({
                "text":doc,
                "title":metadata["title"],
                "relevance":1-distance
            })
        batch_results.append(query_results)
    return batch_results

## 第 6 步：整合——把优化后的 RAG 接入融合 Agent
新建 optimized_hybrid_agent.py：

In [60]:
from dotenv import load_dotenv
import os
import json
import requests

load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
url = "https://openrouter.ai/api/v1/chat/completions"
headers = {
    "authorization":f"Bearer {api_key}",
    "content-type":"application/json"
}

retriever = CachedRetriever()

def optimized_query_knowledge_base(question):
    """
    优化版知识库查询（混合检索 + 缓存）
    """
    results = retriever.hybrid_search(question, top_k=3)
    
    if not results:
        return "知识库中未找到相关信息"
    
    content_parts = []
    for r in results:
        content_parts.append(f"【{r['metadata']['title']}】(相关度：{r['final_score']})\n{r['text']}")
    return "\n\n".join(content_parts)

def get_bom_change(bom_id):
    bom_data = {
        "BOM-1001": "影响物料：螺丝 M3, 钣金件 A12；涉及产线：SMT-2",
        "BOM-1002": "影响物料：电池 5000mAh, 外壳 B 款；涉及产线：组装-3",
    }
    return bom_data.get(bom_id, "未找到该 BOM 的变更记录")


def get_material_stock(material_name):
    stock_data = {
        "螺丝 M3": "库存充足，现有 5000 个",
        "钣金件 A12": "库存紧张，仅剩 120 个，建议补货",
        "电池 5000mAh": "库存充足，现有 2000 个",
        "外壳 B 款": "无库存，已停产",
    }
    return stock_data.get(material_name, f"未找到物料 {material_name} 的库存信息")

# 工具定义
tools =[
    {
        "type": "function",
        "function": {
            "name": "optimized_query_knowledge_base",
            "description": "查询企业内部知识库，包含制度、流程、规范等信息。使用混合检索技术，支持精确匹配和语义匹配。",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {
                        "type": "string",
                        "description": "要查询的问题"
                    }
                },
                "required": ["question"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_bom_change",
            "description": "根据 BOM 编号查询变更影响到的物料和产线",
            "parameters": {
                "type": "object",
                "properties": {
                    "bom_id": {
                        "type": "string",
                        "description": "BOM 编号"
                    }
                },
                "required": ["bom_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_material_stock",
            "description": "查询某个物料的当前库存情况。注意：这是唯一可用的库存查询函数，名称为 get_material_stock，参数为 material_name。禁止使用任何其他名称的函数。",
            "parameters": {
                "type": "object",
                "properties": {
                    "material_name": {
                        "type": "string",
                        "description": "物料名称"
                    }
                },
                "required": ["material_name"]
            }
        }
    }
]

available_functions = {
    "optimized_query_knowledge_base": optimized_query_knowledge_base,
    "get_bom_change": get_bom_change,
    "get_material_stock": get_material_stock
}

class OptimizedHybridAgent:
    """优化版融合 Agent"""
    def __init__(self):
        self.messages = [
            {"role":"system","content":"""你是制造企业的AI助理，你拥有以下能力：
            1.查询企业内部知识库(制度、流程、规范)，使用混合检索，能精准匹配关键词
            2.查询BOM变更影响
            3.查询物料库存
            请根据用户问题灵活选择使用哪些能力，如果需要多方面信息才能回答，请逐步查询。
            """}
        ]
    def chat(self, user_input, max_turns=8):
        self.messages.append({"role":"user", "content":user_input})
        
        for turn in range(max_turns):
            
            data = json.dumps({
                "model":"nvidia/nemotron-3.5-lightning:free",
                "messages":self.messages,
                "tools":tools,
                "tool_choice":"auto"
            })

            response = requests.post(url=url, headers=headers, data=data)
            print(f"第{turn}次,{response.json()}")
            msg = response.json()["choices"][0]["message"]
            self.messages.append(msg)
            
            if not msg.get("tool_calls"):
                return msg["content"]
            
            for tc in msg["tool_calls"]:
                func_name = tc["function"]["name"]
                func_args = json.loads(tc["function"]["arguments"])
            
                if func_name in available_functions:
                    func_to_call = available_functions[func_name]
                    func_results = func_to_call(**func_args)
                else:
                    func_results = f"错误：未找到函数 {func_name}"

                self.messages.append({
                    "role":"tool",
                    "tool_call_id":tc["id"],
                    "content":str(func_results)
            })
        self.messages.append({"role":"user","content":"请基于已有信息给出最终回答。"})
        data = json.dumps({
            "model":"nvidia/nemotron-3.5-lightning:free",
            "messages":self.messages
        })
        response = requests.post(url=url, headers=headers, data=data)
        print(f"最终,{response.json()}")
        return response.json()["choices"][0]["message"]["content"]

# ------------- 测试 ----------------------
# if __name__ == "__main__":
agent = OptimizedHybridAgent()

test_questions = [
    "PLM 系统提交 BOM 变更申请的流程是什么？",
    "密码长度有什么要求？",
    "BOM-1001 变更了，受影响的物料有哪些？库存够吗？"
]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"问题：{q}")
    print(f"{'='*60}")
    answer = agent.chat(q)
    print(f"回答：{answer}")
    agent.messages = [agent.messages[0]]  # 重置，只保留 system prompt


问题：PLM 系统提交 BOM 变更申请的流程是什么？
第0次,{'id': 'gen-1795052201-5sB0fExKgxhfmVknmFrc', 'object': 'chat.completion', 'created': 1795052201, 'model': 'nvidia/nemotron-3.5-lightning:free', 'provider': 'Nvidia', 'system_fingerprint': None, 'service_tier': None, 'choices': [{'index': 0, 'logprobs': None, 'finish_reason': 'tool_calls', 'native_finish_reason': 'tool_calls', 'message': {'role': 'assistant', 'content': None, 'refusal': None, 'reasoning': 'The user asks: "PLM 系统提交 BOM 变更申请的流程是什么？" We need to answer using the capabilities: query internal knowledge base (procedures, rules). Likely answer from knowledge base. Use optimized_query_knowledge_base with question.', 'tool_calls': [{'type': 'function', 'index': 0, 'id': 'call-a0a03a6e-02e1-47e3-9ae5-0d7c99602e30', 'function': {'name': 'optimized_query_knowledge_base', 'arguments': '{"question":"PLM 系统提交 BOM 变更申请的流程"}'}}], 'reasoning_details': [{'type': 'reasoning.text', 'text': 'The user asks: "PLM 系统提交 BOM 变更申请的流程是什么？" We need to answer using th

第0次,{'id': 'gen-1795052221-ScAJQEgzLioOzUdj3kkf', 'object': 'chat.completion', 'created': 1795052221, 'model': 'nvidia/nemotron-3.5-lightning:free', 'provider': 'Nvidia', 'system_fingerprint': None, 'service_tier': None, 'choices': [{'index': 0, 'logprobs': None, 'finish_reason': 'tool_calls', 'native_finish_reason': 'tool_calls', 'message': {'role': 'assistant', 'content': None, 'refusal': None, 'reasoning': 'We need to answer: BOM-1001 changed, what materials affected? And is stock sufficient? So first query BOM change impact using get_bom_change with bom_id "BOM-1001". Then for each affected material, query stock using get_material_stock.\n\nWe need to possibly get list of materials. Then for each, get stock. Then compare? Might need to know required quantity? But we only have stock level, not required quantity. Might just report stock levels. We\'ll proceed.\n\nFirst call get_bom_change.', 'tool_calls': [{'type': 'function', 'index': 0, 'id': 'call-6495fb2f-1343-46ac-8625-6a0704b07

#### 遇到问题1

大模型工具调用的一个核心陷阱：模型并不总是忠实地执行你给出的工具定义，而是会按照它“认为合理”的方式去理解任务。
简单来说，模型调用 query_material_inventory 而不是 get_material_stock，是因为它在“猜测”和“泛化”，而不是在“严格遵循”。  
<b>原因一</b>：模型的“预训练知识”覆盖了你的“即时指令”  
这是最主要的原因。大模型在训练过程中见过海量的代码、API文档和技术文章。在这些数据中，“查询库存”这个功能最常见的函数名是 query_inventory、get_inventory、check_stock 等。像 get_material_stock 这种命名方式虽然正确，但在模型的“经验”里并不是最典型的。
当你问“库存够吗？”时，模型内部的知识检索系统会优先激活它最熟悉的模式——query_material_inventory。它并非故意犯错，而是其强大的“联想”能力压倒了对你提供工具列表的“短期记忆”。这是一种典型的 “知识干扰”。  
<b>原因二</b>：你提供的工具描述存在“模糊性”
你在 tools 定义中为 get_material_stock 提供了这样的描述：
<div style="background-color: #f0f8ff; padding: 10px; border-left: 4px solid #4a90e2;">
  "查询某个物料的当前库存情况"
</div>
这个描述本身没有问题，但它没有足够强的信号来对抗模型内部的固有偏见。你可以对比一下，如果描述改成下面这样，模型会更倾向于使用正确的函数名：<div style="background-color: #f0f8ff; padding: 10px; border-left: 4px solid #4a90e2;">
"查询某个物料的当前库存情况。注意：这是唯一可用的库存查询函数，名称为 get_material_stock，参数为 material_name。禁止使用任何其他名称的函数。"</div>
更强烈的、带有“唯一性”和“排他性”的提示，能有效压制模型的“联想”冲动。
